In [1]:
from langchain.chat_models import init_chat_model, BaseChatModel
from langchain.agents import create_agent
from langchain_core.messages import BaseMessage,AIMessage,SystemMessage,HumanMessage
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Annotated, Literal, Required, NotRequired

In [29]:
model_name = 'google_genai:gemini-3.1-flash-lite'

llm = init_chat_model(model=model_name)

# response = llm.invoke([
#         SystemMessage('You are a helpful assistant'),
#         HumanMessage('What is the capital of france ?')
#     ])

# response = llm.invoke([
#     ('system','You are an helpful assistant'),
#     ('human','What is the capital of france ?')
# ])

llm.invoke([
    {'role':'system','content':'You are a helpful assistant','id':'abc_01'},
    {'role':'human','content':'What is the capital of france ?','id':'abc_01'}
])

response.pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': 'The capital of France is Paris.', 'extras': {'signature': 'EnEKbwFpFH0TZfuI0eYNw1AtgP3FTWyzICcG2/yAHOAkoVKcBxPddccU1p5AUEf6VIVbAeb6Cc8qJbupMLFHQGkvu5kxaw7InqxFuxBgNN/B74PrXw0GNTiC1i8p/7GTgEmSYX3fX/y0A3V8sZfOCMHfQg=='}}]


In [13]:
agent = create_agent(
    model=model_name
)

response = agent.invoke({
    'messages': [
        SystemMessage('You are a helpful assistant'),
        HumanMessage('What is the capital of france')
    ]
})

# for message in response['messages']:
#     print(message)



for block in response['messages'][-1].content_blocks:
    if block['type'] == 'text':
        print(block['text'])

The capital of France is Paris.


In [14]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """
    Add two integers and return their sum.

    Args:
        a: The first integer.
        b: The second integer.

    Returns:
        The sum of a and b.
    """
    add = a + b
    return add

@tool
def sub(a: int, b: int) -> int:
    """
    Sub two integers and return their difference.

    Args:
        a: The first integer.
        b: The second integer.

    Returns:
        The difference of a and b.
    """
    sub = a - b
    return sub


tools = [add,sub]
agent = create_agent(
    model = model_name,
    tools=tools
)

response = agent.invoke({
    'messages':'What is the difference between 100 and 30'
})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

What is the difference between 100 and 30
================================== Ai Message ==================================

[]
Tool Calls:
  sub (call_683205)
 Call ID: call_683205
  Args:
    b: 30
    a: 100
================================= Tool Message =================================
Name: sub

70
================================== Ai Message ==================================

[{'type': 'text', 'text': 'The difference between 100 and 30 is 70.', 'extras': {'signature': 'EnEKbwFpFH0ToNZj5OZ7DHzP1+VOIEAruEh7mj3bwSgs1BH23YiDDRzfjtGbXwxVRRmrFpxDZI2en5pR/DdQFhXHPDtkTly6uMyCgslUYaKDKRxw+kwaK8obLSFjZu1zHLH1Cc7NLGj7myqiFjvxd69fxQ=='}}]


In [16]:
from langgraph.checkpoint.memory import InMemorySaver

cfg1 = {"configurable":{'thread_id':'abc_01'}}
cfg2 = {"configurable":{'thread_id':'abc_02'}}

agent = create_agent(
    model=model_name,
    checkpointer=InMemorySaver()
)

response = agent.invoke({
    'messages' : 'My name is surya'
},cfg1)

for message in response['messages']:
    print(message)


content='My name is surya' additional_kwargs={} response_metadata={} id='436c703c-9d08-4d2e-8ddc-ef72d1d1ca2a'
content=[{'type': 'text', 'text': 'Hello, Surya! It’s nice to meet you. How are you doing today? Is there anything I can help you with?', 'extras': {'signature': 'EnEKbwFpFH0THmZZ8NoM8WUphUPHcso8ow1Krr9/MWsoSqlxeoi1piOT+DsiosffzKCP+6d2E1CTJjY5IYPV1JO+TnIYrzmkmMd1R8E6hUU5rPRN70IVM72L0+XPxbhZXKaIsqVEavceLiCE6ze6xtzOCA=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0b375-c60a-7bf1-91a9-f7cb05dbe547-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 27, 'total_tokens': 33, 'input_token_details': {'cache_read': 0}}


In [17]:
response = agent.invoke({
    'messages' : 'What is my name ?'
},cfg1)

for message in response['messages']:
    print(message)

content='My name is surya' additional_kwargs={} response_metadata={} id='436c703c-9d08-4d2e-8ddc-ef72d1d1ca2a'
content=[{'type': 'text', 'text': 'Hello, Surya! It’s nice to meet you. How are you doing today? Is there anything I can help you with?', 'extras': {'signature': 'EnEKbwFpFH0THmZZ8NoM8WUphUPHcso8ow1Krr9/MWsoSqlxeoi1piOT+DsiosffzKCP+6d2E1CTJjY5IYPV1JO+TnIYrzmkmMd1R8E6hUU5rPRN70IVM72L0+XPxbhZXKaIsqVEavceLiCE6ze6xtzOCA=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0b375-c60a-7bf1-91a9-f7cb05dbe547-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 27, 'total_tokens': 33, 'input_token_details': {'cache_read': 0}}
content='What is my name ?' additional_kwargs={} response_metadata={} id='504e679f-4f9d-4bf4-92ac-1b0f422168d5'
content=[{'type': 'text', 'text': 'Your name is Surya.', 'extras': {'signature': 'EnEKb

In [25]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

store.put(('HR','HYD','SURYA'),'favourites',{'fav1':'car','fav2':'bike'})
store.put(('TECH','BLR','SAMEERA'),'favourites',{'fav1':'Skincare','fav2':'biryani'})

# store.list_namespaces()

# result = store.get(
#     ('TECH', 'BLR', 'SAMEERA'),
#     'favourites'
# )

# print(result.value)

results = store.search(('HR',))

print(results)

[Item(namespace=['HR', 'HYD', 'SURYA'], key='favourites', value={'fav1': 'car', 'fav2': 'bike'}, created_at='2026-09-18T09:08:48.681953+00:00', updated_at='2026-09-18T09:08:48.681959+00:00', score=None)]
